# BIM description preview

This notebook reads Neutral Scene facts and derived matrix evidence. The preview is explanatory only; it does not create facts or scores.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import json
from order_analyzer.test.blender.run_bim_acceptance import run_acceptance
from adapters.ifc_bonsai import load_ifc_neutral_scene
from features.scene_inference import infer_scene_evidence
from features.scene_evaluator import evaluate_scene
from features.neutral_scene_views import analysis_scene_view
from adapters.blender_preview import build_preview_manifest, write_preview_html, write_preview_manifest, write_preview_script

work = Path('test/output/bim_description_demo_work')
artifacts = run_acceptance(work)
description = json.loads(artifacts['description'].read_text(encoding='utf-8'))
room_model = json.loads(artifacts['room_model'].read_text(encoding='utf-8'))
source_ids = description['provenance']['source_ids']
scene = load_ifc_neutral_scene(artifacts['ifc'], analysis_domain_source_id=source_ids['room'])
inference = infer_scene_evidence(scene)
matrix = evaluate_scene(scene)
view = analysis_scene_view(scene)
boundary_channels = [{
    'id': source_id,
    'source_id': source_id,
    'source_geometry_refs': evidence['geometry_refs'],
    'walking': evidence['walking'],
    'visual': evidence['visual'],
    'visible_transmittance': evidence['visible_transmittance'],
    'uncertainties': evidence['uncertainties'],
    'provenance': {'method': 'neutral_scene_channel_evidence'},
} for source_id, evidence in view['channel_evidence'].items()]
preview_matrix = {
    'inferred_regions': inference['inferred_regions'],
    'gap_paths': inference['inferred_region_gap_paths'],
    'boundary_channels': boundary_channels,
}
manifest = build_preview_manifest(scene, preview_matrix)
manifest_path = write_preview_manifest(manifest, Path('test/output/bim_description_demo.preview.json'))
serialized_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
preview_script = write_preview_script(scene, preview_matrix, Path('test/output/bim_description_demo_preview.py'))
preview_html = write_preview_html(serialized_manifest, Path('test/output/bim_description_demo.visual.html'))
print('regions:', len(serialized_manifest['regions']))
print('paths:', len(serialized_manifest['paths']))
print('grid axes:', len(serialized_manifest['grid_axes']))
print('boundary channels:', len(serialized_manifest['boundary_channels']))
print('serialized description modes:', description['input_modes'])
print('serialized room model layers:', sorted(room_model))
print('preview script:', preview_script)
print('preview html:', preview_html)

In [ ]:
from IPython.display import display
import pandas as pd

display(pd.DataFrame([{
    'region_id': region['id'],
    'primary_angle': region['trend_axes'][0]['angle'],
    'primary_strength': region['trend_axes'][0]['strength'],
    'secondary_angle': region['trend_axes'][1]['angle'],
    'secondary_strength': region['trend_axes'][1]['strength'],
} for region in serialized_manifest['regions']]))
display(pd.DataFrame(serialized_manifest['boundary_channels']))

import matplotlib.pyplot as plt
from IPython.display import HTML
display(HTML(preview_html.read_text(encoding='utf-8')))
from math import cos, radians, sin
from matplotlib.patches import Polygon as PlotPolygon
fig, ax = plt.subplots(figsize=(12, 8))
for geometry in serialized_manifest['source_geometry']:
    vertices = geometry['vertices']
    for face in geometry['faces']:
        ax.add_patch(PlotPolygon([[vertices[i][0], vertices[i][1]] for i in face], facecolor='#d5d5d5', edgecolor='#777777', linewidth=0.5))
for path in serialized_manifest['paths']:
    ax.add_patch(PlotPolygon([[p[0], p[1]] for p in path['points']], facecolor='#d946ef', edgecolor='#a21caf', alpha=0.18))
for region in serialized_manifest['regions']:
    points = region['points']
    ax.plot([p[0] for p in points], [p[1] for p in points], color='#16a34a', linewidth=2)
    body = points[:-1] if len(points) > 1 and points[0] == points[-1] else points
    cx = sum(p[0] for p in body) / len(body); cy = sum(p[1] for p in body) / len(body)
    for axis in region['trend_axes']:
        length = 0.7 * max(0.15, axis['strength']); angle = radians(axis['angle'])
        ax.plot([cx-cos(angle)*length, cx+cos(angle)*length], [cy-sin(angle)*length, cy+sin(angle)*length], color='#f97316', linewidth=3)
for axis in serialized_manifest['grid_axes']:
    ax.plot([p[0] for p in axis['curve_points']], [p[1] for p in axis['curve_points']], color='#06b6d4', linewidth=1.2, linestyle='--')
geometry_by_id = {item['geometry_id']: item for item in serialized_manifest['source_geometry']}
for channel in serialized_manifest['boundary_channels']:
    vertices = [v for ref in channel['geometry_refs'] for v in geometry_by_id.get(ref, {}).get('vertices', [])]
    if vertices:
        ax.scatter([sum(v[0] for v in vertices)/len(vertices)], [sum(v[1] for v in vertices)/len(vertices)], color='#dc2626', s=24, zorder=6)
ax.set_aspect('equal'); ax.autoscale_view(); ax.set_title('Source geometry + inferred regions/trends + grid + paths + boundary channels')
ax.set_xlabel('X'); ax.set_ylabel('Y')
fig.savefig('test/output/bim_description_demo.visual.png', dpi=160, bbox_inches='tight')
plt.show()